In [0]:
!pip install httpx polars
!pip install "sidra-fetcher @ git+https://github.com/Quantilica/sidra-fetcher.git"

In [0]:
import httpx
import polars as pl
from sidra_fetcher.sidra import Parametro

# SIDRA no Databricks: população e saneamento

Vamos consultar as **sete tabelas usadas pelo pipeline SINAN–IBGE** e guardar
cada uma em um **DataFrame Spark**. O objetivo desta etapa é conhecer as fontes,
as consultas e as colunas retornadas. As tabelas continuam separadas, com os
valores originais do SIDRA.

| Tabela | Conteúdo | Período | DataFrame |
| --- | --- | --- | --- |
| [136](https://sidra.ibge.gov.br/tabela/136) | População residente do Censo | 2010 | `df136` |
| [4709](https://sidra.ibge.gov.br/tabela/4709) | População residente do Censo | 2022 | `df4709` |
| [6579](https://sidra.ibge.gov.br/tabela/6579) | Estimativas populacionais | Anos selecionados abaixo | `df6579` |
| [3218](https://sidra.ibge.gov.br/tabela/3218) | Água, esgotamento, lixo e energia elétrica | 2010 | `df3218` |
| [6803](https://sidra.ibge.gov.br/tabela/6803) | Abastecimento de água | 2022 | `df6803` |
| [6805](https://sidra.ibge.gov.br/tabela/6805) | Esgotamento sanitário | 2022 | `df6805` |
| [6892](https://sidra.ibge.gov.br/tabela/6892) | Destino do lixo | 2022 | `df6892` |

Os recortes seguem `utils/analysis_pipeline.py`, mas este notebook é independente
dos módulos e dos arquivos locais do projeto. Cada execução consulta o SIDRA ao vivo.

In [0]:
import httpx
from pyspark.sql import Functions as F

In [0]:
# Dados da Tabela 982 do IBGE SIDRA via sidra-fetcher
from sidra_fetcher.sidra import parameter_from_url

url  = "https://apisidra.ibge.gov.br/values/t/3218/n6/all/v/allxp/p/all/c61/all/c299/0/c67/0/c309/0"
param = parameter_from_url(url)

print(param.agregado)   # "1737"
print(param.variaveis)  # ["2265"]
resposta = httpx.get(url, timeout=120)
resposta.raise_for_status()
linhas = resposta.json()

# A primeira linha traz os nomes amigáveis das colunas (/h/y)
df3218= pl.DataFrame(linhas[1:]).rename(linhas[0])


In [0]:
type(df3218)

In [0]:
# Dados da Tabela 982 do IBGE SIDRA via sidra-fetcher
from sidra_fetcher.sidra import parameter_from_url

url  = "https://apisidra.ibge.gov.br/values/t/3218/n6/1100015,1100023,1100049,1100056,1100064,1100031/v/allxp/p/all/c61/0,92853,121290,121294,121295/c299/0/c67/0,1091,2520/c309/0"
param = parameter_from_url(url)

print(param.agregado)   # "1737"
print(param.variaveis)  # ["2265"]
resposta = httpx.get(url, timeout=120)
resposta.raise_for_status()
linhas = resposta.json()

# A primeira linha traz os nomes amigáveis das colunas (/h/y)
df3218= pl.DataFrame(linhas[1:]).rename(linhas[0])

In [0]:
df3218.columns

In [0]:
#print("\n".join(df3218.columns))
print(df3218['Ano'].value_counts() )
print(df3218['Destino do lixo'].value_counts() )
print(df3218['Forma de abastecimento de água'].value_counts() )

print(df3218['Ano'].value_counts() )
df_infra = df3218.select(['Unidade de Medida','Valor','Município','Município (Código)','Variável','Ano',
                         'Forma de abastecimento de água (Código)','Destino do lixo (Código)',
                         'Existência de banheiro ou sanitário e esgotamento sanitário (Código)',
                         'Existência de energia elétrica'])

df_infra.head()

In [0]:
df_infra.row(1)

In [0]:
df_infra_percentual.head(3)
df_infra_filtered = df_infra_percentual.drop(['Unidade de Medida','Ano','Variável'])
df_infra_filtered.head(3)

In [0]:
print(df_infra_filtered['Forma de abastecimento de água (Código)'].value_counts())
print(df_infra_filtered['Destino do lixo (Código)'].value_counts())
print(df_infra_filtered['Existência de energia elétrica'].value_counts())

In [0]:
df_agua = df3218[['Unidade de Medida','Valor','Município','Variável','Ano','Forma de abastecimento de água']]
df_agua.head()
print(df_agua['Variável'].value_counts())

In [0]:
df_infra_absolute = df_infra.filter(pl.col('Unidade de Medida')!='Percentual')
df_infra_absolute.head(3)
print(df_infra_absolute['Forma de abastecimento de água (Código)'].value_counts())
print(df_infra_absolute['Destino do lixo (Código)'].value_counts())

In [0]:
parametro = Parametro(
    agregado="1027",
    territorios={"6": ["all"]},
    variaveis=["all"],
    periodos=["all"],
    classificacoes={},
)

resposta = httpx.get(parametro.url(), timeout=120)
resposta.raise_for_status()
linhas = resposta.json()

# A primeira linha traz os nomes amigáveis das colunas (/h/y)
df1018= pl.DataFrame(linhas[1:]).rename(linhas[0])


In [0]:
print("\n".join(df1018.columns))
print(len(df1018))

In [0]:
df1018.head()


In [0]:
# Dados da Tabela 6579 do IBGE SIDRA via sidra-fetcher
parametro = Parametro(
    agregado="6579",
    territorios={"6": ["all"]},
    variaveis=["all"],
    periodos=["all"],
    classificacoes={},
)

periodos = [
    "2001", "2002", "2003", "2004", "2005", "2006", "2008", "2009", "2011", "2012",
    "2013", "2014", "2015", "2016", "2017", "2018", "2019", "2020", "2021", "2024",
    "2025",
]

quadros = []
for periodo in periodos:
    p = parametro.assign("periodos", [periodo])
    resposta = httpx.get(p.url(), timeout=120)
    resposta.raise_for_status()
    linhas = resposta.json()
    quadros.append(pl.DataFrame(linhas[1:]).rename(linhas[0]))

df6579 = pl.concat(quadros)
print(df6579.head())

In [0]:
print(len(df6579))